In [ ]:
import numpy as np
import tensorflow as tf
import cv2 as cv
import matplotlib.pyplot as plt
import os
import glob

from utils import 

In [ ]:
plt.rcParams['image.cmap'] = 'gray'
plt.rcParams['figure.figsize'] = (9.6, 7.2)

### Load Dataset and Preprocessing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -q drive/My\ Drive/Education/Dataset

In [ ]:

def load_data():
  
  train_size = len(glob.glob('Dataset/images/images_train/*.png'))
  validation_size = len(glob.glob('Dataset/images/images_validation/*.png'))
  test_size = len(glob.glob('Dataset/images/images_test/*.png'))
    
  train_images = [cv.imread('Dataset/images/images_train/' + str(i) + '.png', 0) 
                  for i in range(train_size)]
  train_images = np.stack(train_images)
  
  validation_images = [cv.imread('Dataset/images/images_validation/' + str(i) + '.png', 0) 
                  for i in range(validation_size)]
  validation_images = np.stack(validation_images)
  
  test_images = [cv.imread('Dataset/images/images_test/' + str(i) + '.png', 0) 
                  for i in range(test_size)]
  test_images = np.stack(test_images)
  
  with open('Dataset/formulas/train_formulas.txt') as f:
    train_formulas = [line.split() for line in f]
    
  with open('Dataset/formulas/validation_formulas.txt') as f:
    validation_formulas = [line.split() for line in f]
  
  return train_images, train_formulas, validation_images, validation_formulas, test_images

# loading data
train_images, train_formulas, validation_images, validation_formulas, test_images = load_data()

In [ ]:
def max_len_formulas(all_formulas): 
  return max([len(formula) for formulas in all_formulas for formula in formulas])

def get_fixed_length_formulas(formulas, extra_tokens, max_len):
  
  bof, pad, eof = extra_tokens
  num_formul = len(formulas)
  result = []
  
  for i, theformula in enumerate(formulas):
    formula = theformula.copy()
    # padding
    for _ in range(max_len - len(formula)):
      formula.append(pad)
    # add bof and eof
    formula.insert(0, bof)
    formula.append(eof)
    result.append(formula)
  return result

In [ ]:
# max lenght of all formulas in train and validation
max_len_formula = max_len_formulas([train_formulas, validation_formulas])

extra_tokens = ['__BOF__', '__PAD__', '__EOF__']
# adding pad, bof, eof to train and validation formulas
train_formulas = get_fixed_length_formulas(train_formulas, 
                                extra_tokens, max_len=max_len_formula)
validation_formulas = get_fixed_length_formulas(validation_formulas, 
                                extra_tokens, max_len=max_len_formula)


In [ ]:
# all formulas including train and validation 
all_formulas = train_formulas.copy()
all_formulas.extend(validation_formulas)

tokenizer = tf.keras.preprocessing.text.Tokenizer(filters=None, lower=False)
tokenizer.fit_on_texts(all_formulas)

train_formulas = tokenizer.texts_to_sequences(train_formulas)
validation_formulas = tokenizer.texts_to_sequences(validation_formulas)

train_formulas = np.asarray(train_formulas)
validation_formulas = np.asarray(validation_formulas)

The data is ready. All are in numpy arrays

In [ ]:
# taking one of the training images to check the output shapes 
img = train_images[:32]
img = tf.convert_to_tensor(img, dtype=tf.float32)
img = tf.expand_dims(img, axis=-1)

### CNN

In [ ]:
class CNN(tf.keras.Model):
  
  def __init__(self, filters):
    super(CNN, self).__init__()
    
        
    # c:64, k:(3,3), s:(1,1), p:(1,1) po:(2,2), s:(2,2), p(2,2)
    self.conv1 = tf.keras.layers.Conv2D(filters=filters, 
                                        kernel_size=3, 
                                        strides=1,
                                        padding='same',
                                        activation='relu')
    self.pool1 = tf.keras.layers.MaxPool2D(pool_size=2, 
                                           strides=2, 
                                           padding='same')
    
    # c:128, k:(3,3), s:(1,1), p:(1,1) po:(2,2), s:(2,2), p:(0,0)
    self.conv2 = tf.keras.layers.Conv2D(filters=filters*2, 
                                        kernel_size=3, 
                                        strides=1,
                                        padding='same',
                                        activation='relu')
    self.pool2 = tf.keras.layers.MaxPool2D(pool_size=2, 
                                           strides=2, 
                                           padding='valid')
    
    # c:256, k:(3,3), s:(1,1), p:(1,1), bn -
    self.conv3 = tf.keras.layers.Conv2D(filters=filters*4, 
                                        kernel_size=3, 
                                        strides=1,
                                        padding='same',
                                        activation='relu')
    self.bn1 = tf.keras.layers.BatchNormalization(axis=-1)


    # c:256, k:(3,3), s:(1,1), p:(1,1) po:(2,1), s:(2,1), p(0,0)
    self.conv4 = tf.keras.layers.Conv2D(filters=filters*4, 
                                        kernel_size=3, 
                                        strides=1,
                                        padding='same',
                                        activation='relu')
    self.pool3 = tf.keras.layers.MaxPool2D(pool_size=(2, 1), 
                                           strides=(2, 1), 
                                           padding='valid')


    # c:512, k:(3,3), s:(1,1), p:(1,1), bn po:(1,2), s:(1,2), p:(0,0)
    self.conv5 = tf.keras.layers.Conv2D(filters=filters*8, 
                                        kernel_size=3, 
                                        strides=1,
                                        padding='same',
                                        activation='relu')
    self.pool4 = tf.keras.layers.MaxPool2D(pool_size=(1, 2), 
                                           strides=(1, 2), 
                                           padding='valid')

    # c:512, k:(3,3), s:(1,1), p:(0,0), bn -
    self.conv6 = tf.keras.layers.Conv2D(filters=filters*8, 
                                        kernel_size=3, 
                                        strides=1,
                                        padding='valid',
                                        activation='relu')
    self.bn2 = tf.keras.layers.BatchNormalization(axis=-1)


    
  def call(self, inputs):
    
    
    output = inputs
    for layer in self.layers:
      output = layer(output)
      
    return output
      

In [ ]:
cnn = CNN(32)
print ('CNN output shape: {}'.format(cnn(img).shape))

CNN output shape: (32, 5, 48, 256)


### Encoder

In [ ]:
class Encoder(tf.keras.Model):
  
  def __init__(self, enc_units):
    super(Encoder, self).__init__()
    
    self.gru = tf.keras.layers.Bidirectional(
        tf.keras.layers.GRU(units=enc_units,
                            return_sequences=True, 
                            return_state=True))
    
  def call(self, inputs, hidden=None):
    
    # inputs (batch, height, width, channels)
    height = inputs.shape[1]
    
    outputs = []
    for h in range(height):
      output, _, _ = self.gru(inputs[:,h])
      outputs.append(output)
      
    output = tf.stack(outputs)
    output = tf.transpose(output, [1, 0, 2, 3])
    
    return output
      

In [ ]:
encoder = Encoder(64)
enc_output= encoder(cnn(img))

In [ ]:
print ('Encoder output shape: {}'.format(enc_output.shape))


Encoder output shape: (32, 5, 48, 128)


### Attention

In [ ]:
class Attention(tf.keras.Model):
  
  def __init__(self, units):
    super(Attention, self).__init__()
    
    self.W1 = tf.keras.layers.Dense(units)
    self.W2 = tf.keras.layers.Dense(units)
    self.V = tf.keras.layers.Dense(1)
    
  def call(self, dec_hidden, enc_output):
    
    dec_hidden = tf.expand_dims(dec_hidden, 1)
    dec_hidden = tf.expand_dims(dec_hidden, 1)
        
    score = self.V(tf.keras.activations.tanh(
        self.W1(enc_output) + self.W2(dec_hidden)))
    
    attention_weights = tf.keras.activations.softmax(score, axis=1)

    # context_vector shape after sum == (batch_size, hidden_size)
    context_vector = attention_weights * enc_output
    context_vector = tf.reduce_sum(context_vector, axis=[1, 2])

    return context_vector, attention_weights

In [ ]:
attention_layer = Attention(128)
dec_hidden = tf.random.normal((32, 128))
attention_result, attention_weights = attention_layer(dec_hidden, enc_output)

print("Attention result shape: {}".format(attention_result.shape))
print("Attention weights shape: {}".format(attention_weights.shape))

Attention result shape: (32, 128)
Attention weights shape: (32, 5, 48, 1)


### Decoder

In [ ]:
class Decoder(tf.keras.Model):
  
  def __init__(self, dec_units, embed_size, vocab_size):
    super(Decoder, self).__init__()
    
    self.embedding = tf.keras.layers.Embedding(
        input_dim=vocab_size, 
        output_dim=embed_size)
    
    self.gru = tf.keras.layers.GRU(units=dec_units, 
                      return_sequences=True, return_state=True)
    
    self.fc = tf.keras.layers.Dense(vocab_size)
    
    self.attention = Attention(dec_units)
    
  def call(self, x, dec_state, enc_output, training=True):
    
    context_vector, attention_weights = self.attention(dec_state, enc_output)
    
    x = self.embedding(x)
    
    x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
    
    output, state = self.gru(x)

    output = tf.reshape(output, (-1, output.shape[2]))

    # output shape == (batch_size, vocab)
    output = self.fc(output)
    
    
    return output, state, attention_weights
    

In [ ]:
vocab_size = len(tokenizer.word_index)

decoder = Decoder(128, 128, 569)
dec_output, _, _= decoder(tf.random.uniform((32, 1)), enc_output[:,-1,-1,:], enc_output)
print ('Decoder output shape: {}'.format(dec_output.shape))

Decoder output shape: (32, 569)


### Optimizer and Loss function

In [ ]:
learning_rate = 1e-3
optimizer = tf.keras.optimizers.Adam(learning_rate)
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')

def loss_function(real, pred):
  loss_ = loss_object(real, pred)
  return tf.reduce_mean(loss_)

### Training functions

In [ ]:
@tf.function
def train_step(img, target):
  loss = 0

  with tf.GradientTape() as tape:
    
    cnn_output = cnn(img)
    
    enc_output = encoder(cnn_output)
    
    dec_hidden = enc_output[:,-1,-1,:] 
    
    dec_input = tf.expand_dims(target[:,0], 1)
    
    # Teacher forcing - feeding the target as the next input
    for t in range(1, target.shape[1]):
      # passing enc_output to the decoder
      predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)

      loss += loss_function(target[:, t], predictions)

      # using teacher forcing
      dec_input = tf.expand_dims(target[:,t], 1)

  batch_loss = (loss / int(target.shape[1]))

  variables = cnn.trainable_variables + encoder.trainable_variables + decoder.trainable_variables
  gradients = tape.gradient(loss, variables)
  optimizer.apply_gradients(zip(gradients, variables))
  return batch_loss

In [ ]:
def to_tensor_normalize(img):
  img = tf.convert_to_tensor(img, dtype=tf.float32)
  return (img - 127.5) / 127.5

def generate_batch(batch_size, b, 
                   train_images=train_images, train_formulas=train_formulas):
  
  img = to_tensor_normalize(train_images[b*batch_size:(b+1)*batch_size])
  img = tf.expand_dims(img, -1)
  
  target = tf.convert_to_tensor(train_formulas[b*batch_size:(b+1)*batch_size])
  
  return img, target
  

### Hyperparameters and Initializing modules

In [ ]:
cnn_filters = 8 
enc_units = 16
dec_units = enc_units*2
embedding_dim = 32
vocab_size = len(tokenizer.word_index)

In [ ]:
cnn = CNN(cnn_filters)
encoder = Encoder(enc_units)
decoder = Decoder(dec_units, embedding_dim, vocab_size)

In [ ]:
checkpoint_dir = 'drive/My Drive/temp/training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(optimizer=optimizer,
                                 cnn=cnn,
                                 encoder=encoder,
                                 decoder=decoder)

In [ ]:
batch_size = 32
epochs = 10
steps_per_epoch = train_images.shape[0] // batch_size
print_every = 100
losses = []

### Run 

In [ ]:
for epoch in range(epochs):

  total_loss = 0

  for batch in range(steps_per_epoch):
    img, target = generate_batch(batch_size, batch)
    batch_loss = train_step(img, target)
    total_loss += batch_loss
        
    losses.append(batch_loss.numpy())
    if batch % print_every == 0:
        print('Epoch {} Batch {} Loss {:.4f}'.format(epoch + 1,
                                                     batch,
                                                     np.mean(losses[-print_every:])))

  checkpoint.save(file_prefix = checkpoint_prefix)
  np.save('drive/My Drive/temp/losses', np.array(losses))
  print('Epoch {} Loss {:.4f}'.format(epoch + 1,
                                      total_loss / steps_per_epoch))


Epoch 1 Batch 0 Loss 6.3185


KeyboardInterrupt: ignored

### Evaluation

In [ ]:
# restoring the latest checkpoint in checkpoint_dir
checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

In [ ]:
def toFormula(formula):
  result = ''
  for i in range(1, max_len_formula):
    token = formula[i]
    if token == tokenizer.word_index['__PAD__'] or token == tokenizer.word_index['__EOF__']:
      break
    result += tokenizer.index_word[token]
    result += ' '
  return result[:-1]


def im2latex(img):
  
  img = to_tensor_normalize(img)
  img = tf.expand_dims(img, 0)  
  img = tf.expand_dims(img, -1)
  
  result = []
  
  cnn_output = cnn(img)
  
  enc_output = encoder(cnn_output)
  
  dec_hidden = enc_output[:,-1,-1,:] 
    
  dec_input = tf.expand_dims([tokenizer.word_index['__BOF__']], 1)
  
  for t in range(max_len_formula):
    
    predictions, dec_hidden, attention_weights = decoder(dec_input, dec_hidden, enc_output)

    predicted_token =  tf.argmax(predictions[0]).numpy()
    
    result.append(predicted_token)
    
    dec_input = tf.expand_dims([predicted_token], 1)
    
  return toFormula(result)

In [ ]:
# produce latex from validation images to compute bleu and edit_distance
validation_predicted = []
for img in validation_images:
  validation_predicted.append(im2latex(img))
  
with open('validation_predicted.txt', 'w') as val_file:
  for formula in validation_predicted:
    print(formula, file=val_file)

In [ ]:
!git clone https://github.com/pajouheshgar/DL40959-9798-Project.git -q
!cp -a DL40959-9798-Project/Evaluation/ Evaluation/

In [ ]:
!pip install distance

In [ ]:
print('validation')
!python3 Evaluation/bleu_score.py --target-formulas Dataset/formulas/validation_formulas.txt --predicted-formulas validation_predicted.txt --ngram 5


validation
2000/7391
4000/7391
/usr/local/lib/python3.6/dist-packages/nltk/translate/bleu_score.py:490: UserWarning: 
Corpus/Sentence contains 0 counts of 2-gram overlaps.
BLEU scores might be undesirable; use SmoothingFunction().
  warnings.warn(_msg)
6000/7391
7391/7391
BLEU Score: 0.000048


In [ ]:
!python3 Evaluation/edit_distance.py --target-formulas Dataset/formulas/validation_formulas.txt --predicted-formulas validation_predicted.txt

2000/7391
4000/7391
6000/7391
7391/7391
Edit Distance Accuracy: 0.000001


In [ ]:
# produce latex from test images to compute bleu and edit_distance
test_predicted = []
for img in test_images:
  test_predicted.append(im2latex(img))
  
with open('test_predicted.txt', 'w') as test_file:
  for formula in test_predicted:
    print(formula, file=test_file)